In [69]:
import numpy as np
import itertools

In [70]:
def random_matrix(n, m, p=0.5):
    """
    Generates a random n x m matrix with entries -1, 0, or 1.
    
    Parameters:
    n (int): Number of rows.
    m (int): Number of columns.
    p (float): Probability of an entry being -1 or 1. Default is 0.5.

    Returns:
    A random n x m matrix with entries -1, 0, or 1.
    """
    # check input validity
    if not (isinstance(n, int) and isinstance(m, int) and n > 0 and m > 0):
        raise ValueError("n and m must be positive integers.")
    if not (0 <= p <= 1):
        raise ValueError("p must be between 0 and 1.")
    
    # Generate random numbers, a matrix of shape (n, m) with values in [0, 1)
    random_numbers = np.random.rand(n, m)
    
    # Create the matrix based on the probabilities
    # prob of -1 is p/2, prob of 1 is p/2, prob of 0 is 1 - p
    # x < p/2 gives -1, p/2 <= x < p gives 1, x >= p gives 0
    matrix = np.where(random_numbers < p / 2, -1, 
                      np.where(random_numbers < p, 1, 0))
    
    return matrix

In [73]:
def get_max_delta(A):
    """
    For a square matrix A, find the maximum absolute determinant of all submatrices,
    which is any k x k submatrix for k = 2 to m.
    
    Parameters:
    A (np.ndarray): Input square matrix (m, m).

    Returns:
    float: Maximum absolute determinant of all submatrices.
    """ 
    if not isinstance(A, np.ndarray) or A.shape[0] != A.shape[1]:
        raise ValueError("Input must be a square matrix.")
    
    m = A.shape[0]
    max_det = 0.0
    
    # Iterate over all possible submatrices of size k x k
    # any combination of rows and columns
    # use itertools to generate combinations

    for k in range(2, m + 1):
        row_inx = list(itertools.combinations(range(m), k))
        col_inx = list(itertools.combinations(range(m), k))

        # Generate all k x k submatrices by selecting k rows and k columns
        for rows in row_inx:
            for cols in col_inx:
                submatrix = A[np.ix_(rows, cols)]
                det = np.linalg.det(submatrix)
                max_det = round(max(max_det, abs(det)))
    
    return max_det

A = random_matrix(8, 8, 0.5)
print(A)
max_delta = get_max_delta(A)
print("Maximum absolute determinant of submatrices:", max_delta)

[[ 0  1 -1  0  0  0 -1 -1]
 [-1  0  0  0  1 -1  1 -1]
 [ 0  1  0  1  0  0  0  0]
 [-1  0  1  0 -1  0  0  0]
 [ 1  1  0  0  0  1  0  1]
 [ 0  1  0 -1  0  0  0 -1]
 [ 0  1  1  0  0  1  1  0]
 [ 1  1  1  0  0 -1  0  0]]
Maximum absolute determinant of submatrices: 12


In [74]:
# local search (try to do a motified version of the matrix A on every element)
# A' = A + v * E_ij, make sure A' still only has entries -1, 0, or 1
# see if delta(A') increases, if so, update A
def local_search(A, max_iterations=1000):
    """
    Perform local search to maximize the absolute determinant of submatrices of A.
    
    Parameters:
    A (np.ndarray): Input matrix with entries -1, 0, or 1.
    max_iterations (int): Maximum number of iterations for the local search.

    Returns:
    np.ndarray: Modified matrix with potentially higher maximum absolute determinant.
    """
    if not isinstance(A, np.ndarray) or A.shape[0] != A.shape[1]:
        raise ValueError("Input must be a square matrix.")
    
    m = A.shape[0]
    best_A = A.copy()
    best_delta = get_max_delta(best_A)
    
    for _ in range(max_iterations):
        improved = False # boolean variable to track if improved
        
        for i in range(m):
            for j in range(m):
                # Try to modify A[i, j] to -1, 0, or 1
                original_value = A[i, j]
                for new_value in [-1, 0, 1]:
                    if new_value == original_value:
                        continue
                    
                    # Create a modified copy of A
                    # try the other two values on each element in A to see if it improves delta
                    modified_A = best_A.copy()
                    modified_A[i, j] = new_value
                    
                    # Check if the modified matrix has a higher delta
                    current_delta = get_max_delta(modified_A)
                    
                    if current_delta > best_delta:
                        best_A = modified_A
                        best_delta = current_delta
                        improved = True
        
        if not improved:
            break
    
    return best_A, best_delta

# Example usage of local search
local_search_result = local_search(A)
print("Local Search Result:\n", local_search_result)    

Local Search Result:
 (array([[-1,  1,  1,  1,  1,  1, -1, -1],
       [-1, -1, -1, -1,  1, -1,  1,  1],
       [-1,  1, -1,  1, -1, -1,  1, -1],
       [-1,  1,  1, -1, -1, -1, -1,  1],
       [ 1,  1, -1,  1,  1, -1, -1,  1],
       [ 1,  1, -1, -1,  1,  1,  1, -1],
       [-1,  1,  1,  1, -1,  1,  1,  1],
       [ 1,  1,  1,  1,  1, -1,  1, -1]]), 2176)


I try a 8*8 matrix here, the running time to use this local search strategy is 1m 33s (not bad). I found that after local search, the result matrix only has -1 and 1, and the max(delta(A)) is 4096 (great result). 

Asking gpt, I know there is a Hadamard's inequality to support this result.                           
Hadamard’s inequality states that for any real square matrix 𝐴 ∈ 𝑅^𝑛×𝑛 , the absolute value of its determinant is bounded by the product of the Euclidean norms of its rows:
                                                             ∣det(A)∣≤ ∏ (from i=1 to n) ∥ A_i ∥                                   
where A_i is the i-th row vector of the matrix.

If the matrix A consists only of entries from {−1,1}, and all rows are mutually orthogonal, then the determinant reaches its theoretical maximum:
                                                             ∣det(A)∣=n^(n/2)
 
For n=8, this becomes:                          
∣det(A)∣=8^(8/2) = 4096                                           
​

This local search returns a determinant of 4096, it likely found a matrix that is equivalent to a Hadamard matrix—a matrix over {−1,1}^(8×8) with orthogonal rows. This structure achieves the largest possible determinant.

Local search using the greedy strategy, only accept the change that make delta larger. Once you get to a ‘local high point’, all the changes around you make det lower; it stops there (even if there are many more max_iterations).

In [4]:
# make it a 9*9 matrix
import numpy as np
A = [0,1,1,0,1,0,1,0,1,0,1,1,1,0,0,0,1,0,0,1,0,0,1,1,0,1,0,1,0,1,1,1,1,0,0,0,0,0,1,0,0,1,1,1,1,1,1,0,1,0,1,1,0,0,1,1,1,0,0,0,0,1,0,1,0,0,1,1,0,1,1,0,1,1,0,1,0,1,0,1,1]
A = np.array(A).reshape(9, 9)
print("9x9 Matrix A:\n", A)
print("Maximum absolute determinant of submatrices:", np.linalg.det(A))

9x9 Matrix A:
 [[0 1 1 0 1 0 1 0 1]
 [0 1 1 1 0 0 0 1 0]
 [0 1 0 0 1 1 0 1 0]
 [1 0 1 1 1 1 0 0 0]
 [0 0 1 0 0 1 1 1 1]
 [1 1 0 1 0 1 1 0 0]
 [1 1 1 0 0 0 0 1 0]
 [1 0 0 1 1 0 1 1 0]
 [1 1 0 1 0 1 0 1 1]]
Maximum absolute determinant of submatrices: 92.00000000000001
